# Aero Shield | National Air Intelligence & Bio-Metric Engine
**Lead Engineer:** Sunil Kaimootil | **Version:** v1.8 | **Date:** March 2026

---

### Project Overview
This notebook serves as the data engine for the **Aero Shield** environmental health platform.
It integrates three live APIs to calculate **Shift Cigarette Equivalence** for outdoor workers
in urban India — providing a grounded, math-backed justification for hardware intervention
in high-risk urban environments.

**Pipeline:**
1. **Discovery** — Real-time multi-pollutant harvesting via OpenAQ v3
2. **Physics** — Atmospheric mixing height & ventilation index (stagnation detection)
3. **Mobility** — Traffic functional road class penalties (TomTom Pro)
4. **Morphology** — Urban canyon geometry & population density
5. **Bio-Impact** — Cigarette Equivalence translation (Berkeley Earth methodology)
6. **Strategy** — TAM Index ranking for launch market prioritisation

> All calculation logic lives in `src/models.py`. All API utilities in `src/utils.py`.
> Run `pytest tests/` to verify the math independently of live data.

## Setup

In [ ]:
import sys
import os

# Make src importable from the notebooks/ directory
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster
from IPython.display import display

# Load config (reads from .env — see .env.example)
from src.utils import (
    load_config,
    fetch_pollution_data,
    fetch_weather_data,
    fetch_traffic_data,
    build_morphology_layer,
    build_popup_html,
    get_marker_color,
)
from src.models import (
    calculate_inhaled_dose,
    calculate_cigarette_equivalence,
    calculate_protected_exposure,
    calculate_cigs_prevented,
    calculate_stagnation_penalty,
    calculate_source_penalty,
    calculate_owpei,
    calculate_tam_index,
    classify_toxic_status,
    classify_risk_tier,
    DEFAULT_FILTER_EFFICIENCY,
)

config = load_config()
print(f"Config loaded. Keys present: {list(config.keys())}")

## Block 1: Pollution Layer
Live multi-pollutant harvesting from OpenAQ v3 (India, verified March 2026 readings).

In [ ]:
df_pollution = fetch_pollution_data(
    api_key=config["OPENAQ_API_KEY"],
    country_id=9,
    limit=25,
    target_year_month="2026-03",
)

print(f"\n LAYER 1 COMPLETE: {len(df_pollution)} verified sites with live PM2.5 data.")
display(df_pollution[["location_name", "latitude", "longitude", "pm25", "no2", "o3", "pm10", "last_updated_local"]])

## Block 2: Meteorological Layer
Fetch atmospheric data and calculate Ventilation Index (stagnation detection).

In [ ]:
df_weather = fetch_weather_data(df_pollution, api_key=config["WEATHER_API_KEY"])

print(f"\n LAYER 2 COMPLETE: Meteorological profiles for {len(df_weather)} sites.")
display(df_weather[[
    "location_name", "wind_speed_m_s", "pressure_hpa",
    "mixing_height_m", "ventilation_index", "temperature_c", "humidity_percent"
]])

## Block 3: Urban Mobility Layer
TomTom Pro Traffic — Functional Road Class for diesel-soot source penalties.

In [ ]:
df_traffic = fetch_traffic_data(df_pollution, api_key=config["TOMTOM_API_KEY"])

print(f"\n LAYER 3 COMPLETE: Traffic profiles for {len(df_traffic)} sites.")
display(df_traffic)

## Block 4: Urban Morphology Layer
Urban class classification, Street Aspect Ratios, and population density.

In [ ]:
df_morphology = build_morphology_layer(df_pollution)

print(f"\n LAYER 4 COMPLETE: Morphology profiles for {len(df_morphology)} sites.")
display(df_morphology)

## Block 5: Biomedical Impact Engine
Translate raw air quality data into human-centric health metrics using
the Berkeley Earth cigarette equivalence methodology and Active Worker Persona (25 L/min).

> All math implemented in `src/models.py` — independently verifiable via `pytest tests/`

In [ ]:
# Master join across all four layers
df_master = df_pollution.merge(df_weather, on=["location_name", "last_updated_local"], how="inner")
df_master = df_master.merge(df_traffic, on=["location_name", "last_updated_local"], how="inner")
df_master = df_master.merge(df_morphology, on=["location_name", "last_updated_local"], how="inner")

# Apply penalty models
df_master["stagnation_penalty"] = df_master["ventilation_index"].apply(calculate_stagnation_penalty)
df_master["source_penalty"] = df_master["road_type"].apply(calculate_source_penalty)

# Calculate biomedical impact
df_master["inhaled_dose_ug_hr"] = df_master.apply(
    lambda r: calculate_inhaled_dose(r["pm25"], r["stagnation_penalty"], r["source_penalty"]),
    axis=1,
)
df_master["shift_cigarette_equiv"] = df_master["inhaled_dose_ug_hr"].apply(calculate_cigarette_equivalence)
df_master["protected_cigarette_equiv"] = df_master["shift_cigarette_equiv"].apply(calculate_protected_exposure)
df_master["cigs_prevented_per_shift"] = df_master["shift_cigarette_equiv"].apply(calculate_cigs_prevented)

# OWPEI composite index
df_master["owpei"] = df_master.apply(
    lambda r: calculate_owpei(r["pm25"], r["stagnation_penalty"], r["source_penalty"], r["street_aspect_ratio"]),
    axis=1,
)

# Classification — consistent string values (no emoji in comparison fields)
df_master["toxic_status"] = df_master["pm25"].apply(classify_toxic_status)
df_master["risk_tier"] = df_master["shift_cigarette_equiv"].apply(classify_risk_tier)

print(" LAYER 5 COMPLETE: Biomedical impact calculated.")
display(df_master[[
    "location_name", "pm25", "toxic_status", "risk_tier",
    "shift_cigarette_equiv", "protected_cigarette_equiv", "cigs_prevented_per_shift"
]])

## Block 6: Executive Launch Dashboard
TAM Index ranking — identifies highest-priority launch markets.

In [ ]:
df_master["aero_shield_tam_index"] = df_master.apply(
    lambda r: calculate_tam_index(r["owpei"], r["pop_density_km2"]),
    axis=1,
)

exec_cols = [
    "location_name", "pm25", "risk_tier", "urban_class",
    "shift_cigarette_equiv", "cigs_prevented_per_shift", "aero_shield_tam_index"
]
df_dashboard = df_master[exec_cols].sort_values("aero_shield_tam_index", ascending=False)

print("AERO SHIELD EXECUTIVE LAUNCH DASHBOARD")
print("-" * 90)
display(df_dashboard)

top = df_master.sort_values("aero_shield_tam_index", ascending=False).iloc[0]
weekly = round(top["cigs_prevented_per_shift"] * 6, 1)
print(f"\nSTRATEGIC CONCLUSION:")
print(f"Priority launch site: {top['location_name']}")
print(f"Baseline risk: {top['shift_cigarette_equiv']} cigarettes / 8-hr shift")
print(f"Weekly prevention: ~{weekly} cigarette-equivalents per worker")
print(f"Risk tier: {top['risk_tier']} -> Managed Risk with Aero Shield")

## Block 7: Geospatial Impact Map
Interactive Folium map with popup metrics for each monitoring site.

In [ ]:
m = folium.Map(location=[22.0, 78.0], zoom_start=5, tiles="CartoDB positron")

for _, row in df_master.iterrows():
    color = get_marker_color(row["toxic_status"])  # Uses consistent "TOXIC"/"SAFE" string
    popup_html = build_popup_html(row)

    # Pulsing circle for toxic zones
    if color == "red":
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=14,
            color="red",
            fill=True,
            fill_color="red",
            fill_opacity=0.3,
        ).add_to(m)

    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        icon=folium.Icon(color=color, icon="info-sign"),
        popup=folium.Popup(popup_html, max_width=260),
    ).add_to(m)

print(" BLOCK 7 COMPLETE: Interactive map generated.")
m